In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType, FloatType, BooleanType
import pyspark.sql.functions as F

catalog_name = "ecommerce"

In [0]:
df_silver_order_items = spark.table(f"{catalog_name}.bronze.brz_order_items")

df_silver_order_items.printSchema()

# Transformations

## 1. order_date

In [0]:
# 1.1 Transformation: slv_order_items -> order_date

df_silver_order_items = df_silver_order_items.withColumnRenamed("dt", "order_date")

df_silver_order_items = df_silver_order_items.withColumn("order_date", F.coalesce(
        F.expr("try_to_date(order_date, 'M/d/yy')"),
        F.expr("try_to_date(order_date, 'M/dd/yy')"),
        F.expr("try_to_date(order_date, 'M/d/yyyy')"),
        F.expr("try_to_date(order_date, 'M/dd/yyyy')"),

        F.expr("try_to_date(order_date, 'MM/d/yy')"),
        F.expr("try_to_date(order_date, 'MM/dd/yy')"),
        F.expr("try_to_date(order_date, 'MM/d/yyyy')"),
        F.expr("try_to_date(order_date, 'MM/dd/yyyy')"),

        F.expr("try_to_date(order_date, 'd/M/yy')"),
        F.expr("try_to_date(order_date, 'd/MM/yy')"),
        F.expr("try_to_date(order_date, 'd/M/yyyy')"),
        F.expr("try_to_date(order_date, 'd/MM/yyyy')"),

        F.expr("try_to_date(order_date, 'dd/M/yy')"),
        F.expr("try_to_date(order_date, 'dd/MM/yy')"),
        F.expr("try_to_date(order_date, 'dd/M/yyyy')"),
        F.expr("try_to_date(order_date, 'dd/MM/yyyy')"),

        F.expr("try_to_date(order_date, 'yy/M/d')"),
        F.expr("try_to_date(order_date, 'yy/MM/d')"),
        F.expr("try_to_date(order_date, 'yy/M/dd')"),
        F.expr("try_to_date(order_date, 'yy/MM/dd')"),

        F.expr("try_to_date(order_date, 'yyyy/M/d')"),
        F.expr("try_to_date(order_date, 'yyyy/MM/d')"),
        F.expr("try_to_date(order_date, 'yyyy/M/dd')"),
        F.expr("try_to_date(order_date, 'yyyy/MM/dd')")
    )
)

In [0]:
df_silver_order_items.printSchema()
display(df_silver_order_items.limit(10))

In [0]:
# 1.2 Validation: slv_order_items -> order_date

df_silver_order_items.filter(F.col("order_date").isNull() | (F.col("order_date") > F.current_date()) | (F.col("order_date") < F.lit("2025-01-01").cast(DateType()))).select("order_date", "order_id").show()


valid_dates = spark.read.table(f"{catalog_name}.silver.slv_date_clean").select("date_id")

df_silver_order_items.join(
    valid_dates,
    df_silver_order_items["order_date"] == valid_dates["date_id"],
    how = "left_anti"
).select("order_date").distinct().show()

## 2. order_timestamp

In [0]:
# 2.1 Transformation: slv_order_items -> rename order_timestamp

df_silver_order_items = df_silver_order_items.withColumnRenamed("order_ts", "order_timestamp")

In [0]:
# 2.2 Transformation: slv_order_items -> order_timestamp cast to TimestampType()
df_silver_order_items = df_silver_order_items.withColumn("order_timestamp", F.col("order_timestamp").cast(TimestampType()))

In [0]:
# 2.3 Validation: slv_order_items -> order_timestamp is valid timestamp: Nulls & order_date match

df_silver_order_items.filter(F.col("order_timestamp").isNull() | (F.to_date(F.col("order_timestamp")) != F.col("order_date")) | (F.col("order_timestamp") > F.current_timestamp()) | (F.col("order_timestamp") < F.lit("2025-01-01").cast(TimestampType()))) \
    .select("order_id", "order_date", "order_timestamp").show()

## 3. customer_id

In [0]:
# 3.1 Transformation: slv_order_items -> customer_id




In [0]:
# 3.2 Validation: slv_order_items -> customer_id

df_silver_order_items.filter(
    F.col("customer_id").isNull() |
    (~F.col("customer_id").like("CUST%")) | 
    (F.length(F.col("customer_id")) != 17)
).select("order_id", "item_seq", "customer_id").show()

valid_customers = spark.read.table(f"{catalog_name}.silver.slv_customers_clean").select("customer_id")

df_silver_order_items.join(
    valid_customers,
    on = "customer_id",
    how = "left_anti"
).select("order_id", "item_seq", "customer_id").distinct().show()

## 4. order_id

In [0]:
# 4.1 Transformation: slv_order_items -> order_id

df_silver_order_items = df_silver_order_items.dropDuplicates(["order_id", "item_seq"])


In [0]:
# 4.2 Validation: slv_order_items -> order_id

df_silver_order_items.groupBy("order_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

In [0]:
# 4.3 Validation: check item_seq is sequential per order

from pyspark.sql import Window

win = Window.partitionBy("order_id").orderBy("item_seq")

df_silver_order_items.withColumn("expected_seq", F.row_number().over(win)
).filter(
    F.col("item_seq") != F.col("expected_seq")
).select("order_id", "item_seq", "expected_seq").show()

## 5. item_seq

In [0]:
# 5.1 Transformation: slv_order_items -> item_seq

df_silver_order_items = df_silver_order_items.withColumn(
    "item_seq", 
    F.col("item_seq").cast(IntegerType())
)

In [0]:
# 5.2 Validation: slv_order_items -> item_seq

df_silver_order_items.select("item_seq").distinct().show()

## 6. product_id

In [0]:
# 6.1 Transformation: slv_order_items -> product_id

In [0]:
# 6.2 Validation: slv_order_items -> product_id

df_silver_order_items.filter(
    F.col("product_id").isNull() |
    (F.length(F.col("product_id")) != 13) |
    (F.col("product_id").rlike("[^0-9]"))
).select("order_id", "item_seq", "product_id").show()

valid_products = spark.read.table(f"{catalog_name}.silver.slv_products_clean").select("product_id")

df_silver_order_items.join(
    valid_products,
    on = "product_id",
    how = "left_anti"
).select("product_id").distinct().show()

## 7. quantity

In [0]:
# 7.1 Transformation: slv_order_items -> quantity
df_silver_order_items = df_silver_order_items.withColumn(
    "quantity",
    F.trim(F.initcap(F.col("quantity")))
)

df_silver_order_items = df_silver_order_items.withColumn(
    "quantity",
    F.when(F.col("quantity") == "One", 1)
    .when(F.col("quantity") == "Two", 2)
    .when(F.col("quantity") == "Three", 3)
    .when(F.col("quantity") == "Four", 4)
    .when(F.col("quantity") == "Five", 5)
    .when(F.col("quantity") == "Six", 6)
    .when(F.col("quantity") == "Seven", 7)
    .when(F.col("quantity") == "Eight", 8)
    .when(F.col("quantity") == "Nine", 9)
    .when(F.col("quantity") == "Ten", 10)
    .otherwise(F.col("quantity")) \
        .cast(IntegerType())
)

In [0]:
# 7.2 Validation: slv_order_items -> quantity

df_silver_order_items.select("quantity").distinct().show()

## 8. unit_price_currency

In [0]:
# 8.1 Transformation: slv_order_items -> unit_price_currency

In [0]:
# 8.2 Validation: slv_order_items -> unit_price_currency

valid_currencies = [
    row["currency_code"]
    for row in spark.read.table(f"{catalog_name}.silver.slv_countries_clean")
    .select("currency_code").distinct().collect()
    ]

df_silver_order_items.filter(
    F.col("unit_price_currency").isNull() |
    ~F.col("unit_price_currency").isin(valid_currencies)
).select("order_id", "item_seq", "unit_price_currency").show()

## 9. unit_price

In [0]:
# 9.1 Transformation: slv_order_items -> unit_price

df_silver_order_items = df_silver_order_items.withColumn(
    "unit_price",
    F.trim(F.regexp_replace(F.col("unit_price"), r'[^\d.]', '')).cast("double")
)

display(df_silver_order_items.limit(10))

In [0]:
# 9.2 Validation: slv_order_items -> unit_price | Negative or zero prices; Nulls after cast

df_silver_order_items.filter(F.col("unit_price").isNull() | (F.col("unit_price") <= 0)) \
    .select("order_id", "item_seq", "unit_price") \
    .show()

In [0]:
# 9.3 Validation: Outlier prices (fat finger errors)

df_silver_order_items.select(
    F.min("unit_price").alias("min_price"),
    F.max("unit_price").alias("max_price"),
    F.avg("unit_price").alias("avg_price"),
    F.percentile_approx("unit_price", 0.95).alias("p95_price")
).show()

## 10. discount_pct

In [0]:
# 10.1 Transformation: slv_order_items -> discount_pct

df_silver_order_items = df_silver_order_items.withColumn(
    "discount_pct",
    F.trim(F.regexp_replace(F.col("discount_pct"), r'[^\d.]', '')).cast("double")
)

In [0]:
# 10.2 Validation: slv_order_items -> discount_pct

df_silver_order_items.filter(
    F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") > 100)
).select("order_id", "item_seq", "discount_pct").show()

df_silver_order_items.select(
    F.min("discount_pct").alias("min_discount"),
    F.max("discount_pct").alias("max_discount"),
    F.avg("discount_pct").alias("avg"),
    F.count(F.when(F.col("discount_pct") == 0, 1)).alias("zero_discount_count"),
    F.count(F.when(F.col("discount_pct") > 50, 1)).alias("high_discount_count")
).show()

## 11. tax_amount

In [0]:
# 11.1 Transformation: slv_order_items -> tax_amount

df_silver_order_items = df_silver_order_items.withColumn(
    "tax_amount",
    F.regexp_replace(F.col("unit_price"), r'[^\d.]', '').cast("double")
)

In [0]:
# 11.2 Validation: slv_order_items -> tax_amount

df_silver_order_items.filter(
    F.col("tax_amount").isNull() | (F.col("tax_amount") < 0) | (F.col("tax_amount") > F.col("unit_price"))
).select(
    "order_id", "unit_price", "tax_amount"
).show()

## 12. channel

In [0]:
# 12.1 Transformation: slv_order_items -> channel

df_silver_order_items = df_silver_order_items.withColumn(
    "channel", F.lower(F.trim(F.col("channel")))
)

df_silver_order_items = df_silver_order_items.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel"))
)

In [0]:
# 12.2 Validation: slv_order_items -> channel

valid_channels = ['Website', 'Mobile']

df_silver_order_items.filter(
    F.col("channel").isin(valid_channels)) \
    .select("channel").distinct().show()

## 13. coupon_code

In [0]:
# 13.1 Transformation: slv_order_items -> coupon_code

df_silver_order_items = df_silver_order_items.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

In [0]:
# 13.2 Validation: slv_order_items -> coupon_code

df_silver_order_items.select("coupon_code").distinct().show()

## 14. processed_time

In [0]:
df_silver_order_items = df_silver_order_items.withColumn(
  "processed_time", F.current_timestamp()
)

# Clean vs Quarantined Data

In [0]:
df_silver_order_items_clean = df_silver_order_items.filter(
    F.col("order_date").isNotNull() & (F.col("order_date") <= F.current_date()) & (F.col("order_date") >= F.lit("2025-01-01").cast(DateType())) & (F.col("order_date").isin(valid_dates)) &
    F.col("customer_id").isNotNull() & (F.col("customer_id").like("CUST%")) & (F.length(F.col("customer_id")) == 17) & (F.col("customer_id").isin(valid_customers)) &
    F.col("order_id").isNotNull() &
    F.col("item_seq").isNotNull() & (F.col("item_seq") > 0) &
    F.col("product_id").isNotNull() & (F.length(F.col("product_id")) == 13) & (F.col("product_id").isin(valid_products)) &
    F.col("quantity").isNotNull() & (F.col("quantity") > 0) &
    F.col("unit_price_currency").isNotNull() & (F.col("unit_price_currency").isin(valid_currencies)) &
    F.col("unit_price").isNotNull() & (F.col("unit_price") > 0) &
    F.col("discount_pct").isNotNull() & (F.col("discount_pct") >= 0) & (F.col("discount_pct") <= 100) &
    F.col("tax_amount").isNotNull() & (F.col("tax_amount") >= 0) & (F.col("tax_amount") <= F.col("unit_price")) &
    F.col("channel").isNotNull() & (F.col("channel").isin(valid_channels)) 
)

df_silver_order_items_quarantine = df_silver_order_items.filter(
    F.col("order_date").isNull() | (F.col("order_date") > F.current_date()) | (F.col("order_date") < F.lit("2025-01-01").cast(DateType())) | (~F.col("order_date").isin(valid_dates)) |
    F.col("customer_id").isNull() | (~F.col("customer_id").like("CUST%")) | (F.length(F.col("customer_id")) != 17) | (~F.col("customer_id").isin(valid_customers)) |
    F.col("order_id").isNull() |
    F.col("item_seq").isNull() | (F.col("item_seq") <= 0) |
    F.col("product_id").isNull() | (F.length(F.col("product_id")) != 13) | (~F.col("product_id").isin(valid_products)) |
    F.col("quantity").isNull() | (F.col("quantity") <= 0) |
    F.col("unit_price_currency").isNull() | (~F.col("unit_price_currency").isin(valid_currencies)) |
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0) |
    F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") > 100) |
    F.col("tax_amount").isNull() | (F.col("tax_amount") < 0) | (F.col("tax_amount") > F.col("unit_price")) |
    F.col("channel").isNull() | (~F.col("channel").isin(valid_channels))) \
        .withColumn(
            "rejection_reason",
            F.when(F.col("order_date").isNull() | (F.col("order_date") > F.current_date()) | (F.col("order_date") < F.lit("2025-01-01").cast(DateType())) | (~F.col("order_date").isin(valid_dates)), "null/invalid order_date")
            
            .when(F.col("customer_id").isNull() | (~F.col("customer_id").like("CUST%")) | (F.length(F.col("customer_id")) != 17) | (~F.col("customer_id").isin(valid_customers)), "null/invalid customer_id")
            
            .when(F.col("order_id").isNull(), "null order_id")
            .when(F.col("item_seq").isNull() | (F.col("item_seq") <= 0), "null/invalid item_seq")
            .when(F.col("product_id").isNull() | (F.length(F.col("product_id")) != 13) | (~F.col("product_id").isin(valid_products)), "null/invalid product_id")
            .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "null/invalid quantity")
            .when(F.col("unit_price_currency").isNull() | (~F.col("unit_price_currency").isin(valid_currencies)), "null/invalid unit_price_currency")
            .when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), "null/invalid unit_price")
            .when(F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") > 100), "null/invalid discount_pct")
            .when(F.col("tax_amount").isNull() | (F.col("tax_amount") < 0) | (F.col("tax_amount") > F.col("unit_price")), "null/invalid tax_amount")
            .when(F.col("channel").isNull() | (~F.col("channel").isin(valid_channels)), "null/invalid channel")
            )
            


# Write to Delta

In [0]:
df_silver_order_items_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_order_items_clean")

df_silver_order_items_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_order_items_quarantine")